## Jumia Nigeria Phone data web-scrape 

In [1]:
# importing libraries

from bs4 import BeautifulSoup
import requests 
import smtplib
import time
import datetime
import numpy as np

import pandas as pd

In [2]:
url = "https://www.jumia.com.ng/catalog/?q=phones"

In [3]:
#headers for requests

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

In [4]:
# HTTP Requests

requests.get(url, headers = headers)

<Response [200]>

In [5]:
page = requests.get(url, headers = headers)

In [6]:
type(page.content)

bytes

In [7]:
## Soup objects containing all data
soup = BeautifulSoup(page.content, "html.parser")

In [8]:
links = soup.find_all('a', class_ = "core")

In [9]:
link = links[0].get('href')

In [16]:
product_list = "https://www.jumia.com.ng" + link

In [17]:
product_list

'https://www.jumia.com.ng/xiaomi-redmi-a5-6.88-4gb-ram128gb-rom-black-408237971.html'

In [18]:
new_webpage = requests.get(product_list, headers = headers)

In [19]:
print(product_list)

https://www.jumia.com.ng/xiaomi-redmi-a5-6.88-4gb-ram128gb-rom-black-408237971.html


In [20]:
new_webpage

<Response [200]>

In [22]:
new_soup = BeautifulSoup(new_webpage.content, 'html.parser')

In [23]:
new_soup

<!DOCTYPE html>
<html dir="ltr" lang="en"><head><meta charset="utf-8"/><title>XIAOMI REDMI A5 -  6.88   4GB RAM/128GB ROM  -- BLACK | Jumia Nigeria</title><meta content="product" property="og:type"/><meta content="Jumia Nigeria" property="og:site_name"/><meta content="REDMI A5 -  6.88   4GB RAM/128GB ROM  -- BLACK" property="og:title"/><meta content='Processor : UNISOC T7250 processor12nm process, octa-core CPU: Up to 1.8GHzDimensions :Height: 171.7mmWidth: 77.8mmThickness: 8.26mmWeight: 193g*Data obtained from Xiaomi Internal Labs. Industry measurement methods may vary, and therefore actual results may differ.Display:6.88\" large screen display1640*720, 260 ppi Contrast ratio: 1500:1Color depth: 8-bitColor gamut: 70% NTSC Brightness: 450 nits (typ)Refresh rate: Up to 120Hz**Refresh rate can be adjusted to up to 120Hz for supported apps. DC dimmingRear Camera :film Camera | HDR mode | Ultra HD | Night mode32MP main camera4P lens f/2.0Auxiliary lens Rear camera video recording1080p 1920

In [34]:
new_soup.find('h1', attrs={"class": '-fs20 -pts -pbxs'}).get_text().split(',')[0].strip()

'XIAOMI REDMI A5 -  6.88   4GB RAM/128GB ROM  -- BLACK'

In [36]:
new_soup.find('span', attrs={"class": '-b -ubpt -tal -fs16'}).get_text()

'₦ 128,468'

In [37]:
new_soup.find('span', attrs={"class": '-tal -gy5 -ubpt -lthr -fs12'}).get_text()

'₦ 131,172'

In [41]:
new_soup.find('div', attrs={"class": 'stars _m _al'}).get_text()

'4 out of 5'

In [43]:
new_soup.find('a', attrs={"class": '_more'}).get_text()

'XIAOMI'

In [45]:
new_soup.find('a', attrs={"class": '-plxs _more'}).get_text()

'(2010 verified ratings)'

In [55]:
new_soup.find('div', attrs={"class": 'stars _m _al'}).get_text()

'4 out of 5'

### Automated Function to scrape the data

In [63]:
def get_title(soup):
    
    try:
        # outer tag object
        title = soup.find('h1', attrs = {"class": '-fs20 -pts -pbxs'})
        
        title_value = title.text
        
        #title as a string value
        title_string = title_value.strip()
        
    except AttributeError:
        title_string = ""
        
    return title_string


#Funtion to extract Product Brand
def get_brand(soup):
    
    try: brand = soup.find("a", attrs = {"class" : '_more'}).get_text().strip()
        
    except AttributeError:
        brand = ""
        
    return brand


# Function to extract product price
def get_price(soup):
    
    try:
        price = soup.find('span', attrs={"class": '-b -ubpt -tal -fs16'}).get_text().strip()
        
    except AttributeError:
        price = ""
            
    return price


# Function to extract old product price
def get_old_price(soup):
    
    try:
        old_price = soup.find('span', attrs={"class": '-tal -gy5 -ubpt -lthr -fs12'}).get_text().strip()
        
    except AttributeError:
        old_price = ""
            
    return old_price


# Function to extract Product Rating
def get_rating(soup):
    
    try:
        rating = soup.find('div', attrs = {"class" : 'stars _m _al'}).get_text().strip()
    
    except AttributeError:
        rating = ""
            
    return rating


# Function to extract Number of User Reviews 
def get_review_count(soup):
    
    try:
        review_count = soup.find("a", attrs = {'class' : '-plxs _more'}).get_text().strip()
        
    except AttributeError:
        review_count = ""
    
    return review_count




In [48]:
today = datetime.date.today()

print(today)

2026-02-22


In [65]:

if __name__ == "__main__":
    
    #headers for request
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    today = datetime.date.today()


    df = {"Today":[], "Title":[], "Brand":[], "Price":[], "Old Price":[], "Rating":[], "Reviews":[]}

    # To loop through 8 pages
    for page_num in range(1,10):
        print(f"Scraping page {page_num}...")

        # Webpage URL
        url = f"https://www.jumia.com.ng/catalog/?q=phones&page={page_num}" 
        print(f"url: {url}")
        
        # HTTP Requests
        page = requests.get(url, headers = headers)

        ## Soup objects containing all data
        soup = BeautifulSoup(page.content, "html.parser")

        # fetch links as list of tag objects 
        links = soup.find_all('a', attrs = {'class':"core"})

        # Store the Links
        links_list = []

        # Loop for extracting links from Tag objects 
        for link in links:
                links_list.append(link.get('href'))


        # Loop for extracting product details from each link
        for link in links_list:
            new_webpage = requests.get("https://www.jumia.com.ng" + link, headers=headers)

            new_soup = BeautifulSoup(new_webpage.content, "html.parser")

            # Function calls to display all necessary product information
            df["Today"].append(today)
            df["Title"].append(get_title(new_soup))
            df["Brand"].append(get_brand(new_soup))
            df["Price"].append(get_price(new_soup))
            df["Old Price"].append(get_old_price(new_soup))
            df["Rating"].append(get_rating(new_soup))
            df["Reviews"].append(get_review_count(new_soup))
            
            
            time.sleep(3)  
    
    
    jumia_df = pd.DataFrame.from_dict(df)
    jumia_df["Title"].replace('', np.nan, inplace = True)
    jumia_df = jumia_df.dropna(subset=['Title'])
    jumia_df.to_csv("jumia_data.csv", header =True, index = False)

    print("Done 👍")

Scraping page 1...
url: https://www.jumia.com.ng/catalog/?q=phones&page=1
Scraping page 2...
url: https://www.jumia.com.ng/catalog/?q=phones&page=2
Scraping page 3...
url: https://www.jumia.com.ng/catalog/?q=phones&page=3
Scraping page 4...
url: https://www.jumia.com.ng/catalog/?q=phones&page=4
Scraping page 5...
url: https://www.jumia.com.ng/catalog/?q=phones&page=5
Scraping page 6...
url: https://www.jumia.com.ng/catalog/?q=phones&page=6
Scraping page 7...
url: https://www.jumia.com.ng/catalog/?q=phones&page=7
Scraping page 8...
url: https://www.jumia.com.ng/catalog/?q=phones&page=8
Scraping page 9...
url: https://www.jumia.com.ng/catalog/?q=phones&page=9
Done 👍


In [66]:
# Exporting file as CSV

jumia_df.to_csv(r"C:\Users\ikeol\OneDrive\Documents\Web-Scraping\Data\Jumia Phones data.csv")

print("File exported Successfully! 👍")

File exported Successfully! 👍


In [64]:
# Ignore


print(get_title(new_soup))
print(get_price(new_soup))
print(get_old_price(new_soup))
print(get_rating(new_soup))
print(get_review_count(new_soup))

XIAOMI REDMI A5 -  6.88   4GB RAM/128GB ROM  -- BLACK
₦ 128,468
₦ 131,172
4 out of 5
(2010 verified ratings)
